# GPT-Neo inference with the HF's Transformers Library
This notebook is a companion of chapter 4 of the "Domain Specific LLMs in Action" book, author Guglielmo Iozzia, [Manning Publications](https://www.manning.com/), 2024.  
The code in this notebook is to introduce readers to the inference (text generation) with the [GPT-Neo model](https://github.com/EleutherAI/gpt-neo) using the Hugging Face's [Transformers library](https://github.com/huggingface/transformers). It can be executed in the Colab free tier with hardware acceleration (GPU).  
More details about the code can be found in the book's chapter.

Install the missing requirements in the Colab VM (HF's Accelerate only).

In [2]:
!pip install -q accelerate

Download the GPT-Neo 2.7B model and the associated tokenizer from the HF's Hub. The model is loaded in full precision and is then loaded into the GPU.

In [3]:
import torch
from transformers import GPTNeoForCausalLM, GPT2Tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_id = "EleutherAI/gpt-neo-2.7B"
tokenizer = GPT2Tokenizer.from_pretrained(model_id)
model = GPTNeoForCausalLM.from_pretrained(model_id, device_map="auto")
model.to(device)

tokenizer_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/10.7G [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.
You shouldn't move a model that is dispatched using accelerate hooks.


RuntimeError: You can't move a model that has some modules offloaded to cpu or disk.

Verify where the model layers have been loaded (all in the GPU memory or also RAM and/or disk).

In [4]:
model.hf_device_map

{'transformer.wte': 'cpu',
 'lm_head': 'cpu',
 'transformer.wpe': 'cpu',
 'transformer.drop': 'cpu',
 'transformer.h.0': 'cpu',
 'transformer.h.1': 'cpu',
 'transformer.h.2': 'cpu',
 'transformer.h.3': 'cpu',
 'transformer.h.4': 'cpu',
 'transformer.h.5': 'cpu',
 'transformer.h.6': 'cpu',
 'transformer.h.7': 'cpu',
 'transformer.h.8': 'cpu',
 'transformer.h.9': 'cpu',
 'transformer.h.10': 'cpu',
 'transformer.h.11': 'cpu',
 'transformer.h.12': 'cpu',
 'transformer.h.13': 'cpu',
 'transformer.h.14': 'cpu',
 'transformer.h.15': 'disk',
 'transformer.h.16': 'disk',
 'transformer.h.17': 'disk',
 'transformer.h.18': 'disk',
 'transformer.h.19': 'disk',
 'transformer.h.20': 'disk',
 'transformer.h.21': 'disk',
 'transformer.h.22': 'disk',
 'transformer.h.23': 'disk',
 'transformer.h.24': 'disk',
 'transformer.h.25': 'disk',
 'transformer.h.26': 'disk',
 'transformer.h.27': 'disk',
 'transformer.h.28': 'disk',
 'transformer.h.29': 'disk',
 'transformer.h.30': 'disk',
 'transformer.h.31': 'dis

Perform standard inference (text completion).

In [5]:
prompt = "The story so far: in the beginning, the universe was created."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

generated_ids = model.generate(input_ids,
                               do_sample=True,
                               temperature=0.9,
                               max_length=200,
                               pad_token_id=50256)
generated_text = tokenizer.decode(generated_ids[0])
print(generated_text)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


The story so far: in the beginning, the universe was created. It had stars. And it had planets.

The most important of these was Earth. It was a very special planet. It was the perfect place. It was the center of creation. It was home to all living things. It was the center of the universe. The perfect home for all humankind.

But things were not perfect. There were many things that were not perfect. There were too many things. It seemed as though everything had to be perfect. Everything had to be right. Everything had to have the proper color. Everything had to have the proper shape. Everything had to be the right size.

And there were too many humans on Earth.

In one day, there were more humans on Earth than there were stars. That is just one day, you know. And there are millions of humans on Earth.

There were too many humans.

And things were not too


Do few-shot text classification (the model can generalize learning from few new and unseen examples.

In [6]:
prompt = """
Sentence: This movie is very nice.
Sentiment: positive

#####

Sentence: I hated this movie, it sucks.
Sentiment: negative

#####

Sentence: This movie was actually pretty funny.
Sentiment: positive

#####

Sentence: This movie could have been better.
Sentiment: neutral
"""
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

generated_ids = model.generate(input_ids,
                               do_sample=True,
                               temperature=0.9,
                               max_length=200,
                               pad_token_id=50256)
generated_text = tokenizer.decode(generated_ids[0])
print(generated_text)


Sentence: This movie is very nice.
Sentiment: positive

#####

Sentence: I hated this movie, it sucks.
Sentiment: negative

#####

Sentence: This movie was actually pretty funny.
Sentiment: positive

#####

Sentence: This movie could have been better.
Sentiment: neutral

#####

Sentence: This movie is boring.
Sentiment: positive

#####

Sentence: This movie is actually good.
Sentiment: neutral

#####

Sentence: This movie is good.
Sentiment: positive

#####

Sentence: This movie was actually really good.
Sentiment: neutral

#####

Sentence: This movie was good.
Sentiment: neutral

#####

Sentence: This movie was not good.
Sentiment: positive

<|endoftext|>


Do Python code generation.

In [ ]:
prompt = """Instruction: Generate a Python function that lets you reverse a list of integers.

Answer: """
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

generated_ids = model.generate(input_ids,
                               do_sample=True,
                               temperature=0.9,
                               max_length=200,
                               pad_token_id=50256
                               )
generated_text = tokenizer.decode(generated_ids[0])
print(generated_text)

Do batch text completion.

In [ ]:
texts = ["Once there was a man ", "The weather today will be ", "A great soccer player must "]

tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
encoding = tokenizer(texts, padding=True, return_tensors='pt').to(device)
with torch.no_grad():
    generated_ids = model.generate(**encoding,
                                   do_sample=True,
                                   temperature=0.9,
                                   max_length=50,
                                   pad_token_id=50256)
generated_texts = tokenizer.batch_decode(
    generated_ids, skip_special_tokens=True)

for text in generated_texts:
  print("---------")
  print(text)

Benchmarking the model on text completion: comparing the cases where the KV cache is used to those where it isn't.

In [ ]:
import time
import numpy as np

prompt = "The story so far: in the beginning, the universe was created."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

for use_cache in (True, False):
  times = []
  for _ in range(20):
    start = time.time()
    generated_ids = model.generate(input_ids,
                                  do_sample=True,
                                  temperature=0.9,
                                  max_length=200,
                                  pad_token_id=50256,
                                  use_cache=use_cache)
    times.append(time.time() - start)
  print(f"{'Using' if use_cache else 'No'} KV cache: {round(np.mean(times), 3)} +- {round(np.std(times), 3)} seconds")

Benchmarking the model's total generation time.

In [ ]:
import time
import numpy as np

prompt = "The story so far: in the beginning, the universe was created."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

max_length = 300
times = []
inference_runs = 21
for _ in range(inference_runs):
  start = time.time()
  generated_ids = model.generate(input_ids,
                                do_sample=True,
                                temperature=0.9,
                                max_length=max_length,
                                pad_token_id=50256,
                                )
  times.append(time.time() - start)
print(f"Average Total Generation time: {round(np.mean(times[1:]), 3)} +- {round(np.std(times[1:]), 3)} seconds")